In [ ]:
import numpy as np
from stl import mesh
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from stl import mesh as stl_mesh
from pathlib import Path
from hac.minkowski import minkowski_reconstruct, build_convex_mesh, save_stl
from hac.eval import calculate_voxel_measure
import trimesh
import pyvista as pv
pv.set_jupyter_backend('html')

## Test 1: Icosphere:

In [ ]:
def create_icosphere(subdivisions: int = 2):
    """Geodesic icosphere with `4^subdivisions * 20` faces, all projected to unit sphere."""
    phi = (1 + np.sqrt(5)) / 2
    raw = np.array([
        [-1,  phi, 0], [ 1,  phi, 0], [-1, -phi, 0], [ 1, -phi, 0],
        [ 0, -1,  phi], [ 0,  1,  phi], [ 0, -1, -phi], [ 0,  1, -phi],
        [ phi, 0, -1], [ phi, 0,  1], [-phi, 0, -1], [-phi, 0,  1]
    ], dtype=np.float64)
    verts = list(raw / np.linalg.norm(raw[0]))

    faces = [
        [0,11,5],[0,5,1],[0,1,7],[0,7,10],[0,10,11],
        [1,5,9],[5,11,4],[11,10,2],[10,7,6],[7,1,8],
        [3,9,4],[3,4,2],[3,2,6],[3,6,8],[3,8,9],
        [4,9,5],[2,4,11],[6,2,10],[8,6,7],[9,8,1]
    ]

    def midpoint(p1, p2, cache):
        key = (min(p1, p2), max(p1, p2))
        if key not in cache:
            m = (verts[p1] + verts[p2]) / 2
            m = m / np.linalg.norm(m)
            cache[key] = len(verts)
            verts.append(m)
        return cache[key]

    for _ in range(subdivisions):
        new_faces, cache = [], {}
        for a, b, c in faces:
            ab, bc, ca = midpoint(a,b,cache), midpoint(b,c,cache), midpoint(c,a,cache)
            new_faces += [[a,ab,ca],[b,bc,ab],[c,ca,bc],[ab,bc,ca]]
        faces = new_faces

    V = np.array(verts)
    F = np.array(faces)

    # Face normals and areas
    e1 = V[F[:,1]] - V[F[:,0]]
    e2 = V[F[:,2]] - V[F[:,0]]
    cross = np.cross(e1, e2)
    mag   = np.linalg.norm(cross, axis=1)
    face_normals = cross / mag[:, None]
    face_areas   = 0.5 * mag
    return V, F, face_normals, face_areas


sphere_verts, sphere_faces, normals, sigma = create_icosphere(subdivisions=3)
out_p = "/zhome/95/1/155570/HAC-DTU/workspaces/christian/stls/sphere.stl"
save_stl(sphere_verts, sphere_faces, out_p)

vertices_cloud = minkowski_reconstruct(sigma, normals, verbose=True,epsilon=0.001)
mesh_verts, mesh_faces = build_convex_mesh(vertices_cloud)
out_p2 = "/zhome/95/1/155570/HAC-DTU/workspaces/christian/stls/sphere_rec.stl"
save_stl(mesh_verts, mesh_faces, out_p2)
print(f"Reconstructed convex mesh: {len(mesh_verts)} vertices, {len(mesh_faces)} faces")


sph = pv.read(out_p)
sph.plot(notebook=True,window_size=[500,500],show_edges=True)

sph = pv.read(out_p2)
sph.plot(notebook=True,window_size=[500,500],show_edges=True)
print(f"True mesh: {len(sphere_verts)} vertices, {len(normals)} faces")
print(f"Reconstructed mesh: {len(sphere_verts)} vertices, {len(normals)} faces")
print(f"Error: {calculate_voxel_measure(trimesh.load_mesh(out_p),trimesh.load_mesh(out_p2))}")

  iter    25: volume=2.048073 alphak=0.003093
Converged after 31 iterations: alphak=0.000773
Reconstructed convex mesh: 642 vertices, 1280 faces


EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

True mesh: 642 vertices, 1280 faces
Reconstructed mesh: 642 vertices, 1280 faces
Error: 0.9952798703156289


## Test 2: Asteroid 2

In [10]:
from hac.data import get_model_stl_path, load_data

stl_path = str(get_model_stl_path(2))
shape = mesh.Mesh.from_file(stl_path)
vectors = shape.vectors.astype(np.float16)  # (n_facets, 3, 3)
edge1 = vectors[:, 1, :] - vectors[:, 0, :]  # (n_facets, 3)
edge2 = vectors[:, 2, :] - vectors[:, 0, :]

cross = np.cross(edge1, edge2)  # (n_facets, 3)
norm_mag = np.linalg.norm(cross, axis=1, keepdims=True)
normals = cross / norm_mag  # (n_facets, 3)  unit normals
sigma = 0.5 * norm_mag.squeeze(1)  # (n_facets,)

print(f"True mesh: {len(normals)} faces")
sph = pv.read(stl_path)
sph.plot(notebook=True,window_size=[500,500],show_edges=True)

# asteroid2.stl is a 538k-triangle mesh of what's geometrically an exact
# convex polyhedron (a cube with one corner sawed off) -- its true shape
# needs only its own convex hull, which has 16 triangles / 10 vertices.
asteroid_mesh = trimesh.load_mesh(stl_path)
hull = asteroid_mesh.convex_hull
simplified_path = "/zhome/95/1/155570/HAC-DTU/workspaces/christian/stls/asteroid2_simplified.stl"
hull.export(simplified_path)
print(f"Simplified mesh: {len(hull.faces)} faces, {len(hull.vertices)} vertices")
print(f"Voxel similarity vs true mesh: {calculate_voxel_measure(asteroid_mesh, trimesh.load_mesh(simplified_path))}")
sph = pv.read(simplified_path)
sph.plot(notebook=True,window_size=[500,500],show_edges=True)

True mesh: 537950 faces


EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

Simplified mesh: 16 faces, 10 vertices
Voxel similarity vs true mesh: 0.9965720363939059


EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

In [8]:
simplified_shape = mesh.Mesh.from_file(simplified_path)
vectors = simplified_shape.vectors.astype(np.float64)
edge1 = vectors[:, 1, :] - vectors[:, 0, :]
edge2 = vectors[:, 2, :] - vectors[:, 0, :]
cross = np.cross(edge1, edge2)
norm_mag = np.linalg.norm(cross, axis=1, keepdims=True)
tri_normals = cross / norm_mag
tri_sigma = 0.5 * norm_mag.squeeze(1)

# minkowski_reconstruct expects one facet per unique plane, but an STL
# triangulates each flat face into 2+ triangles sharing the same normal.
# Duplicate normals make the dual points coincide, which stalls the solver
# (alphak plateaus and it never converges) -- so merge coplanar triangles
# into single facets, summing their area, before reconstructing.
rounded = np.round(tri_normals, 6)
normals, inverse = np.unique(rounded, axis=0, return_inverse=True)
sigma = np.zeros(len(normals))
np.add.at(sigma, inverse, tri_sigma)
print(f"Simplified mesh triangles: {len(tri_normals)} -> unique facets: {len(normals)}")

vertices_cloud = minkowski_reconstruct(sigma, normals, verbose=True, epsilon=0.0001)
mesh_verts, mesh_faces = build_convex_mesh(vertices_cloud)
out_p2 = "/zhome/95/1/155570/HAC-DTU/workspaces/christian/stls/ast2_mink_rec.stl"
save_stl(mesh_verts, mesh_faces, out_p2)
print(f"Reconstructed convex mesh: {len(mesh_verts)} vertices, {len(mesh_faces)} faces")

sph = pv.read(simplified_path)
sph.plot(notebook=True,window_size=[500,500],show_edges=True)

sph = pv.read(out_p2)
sph.plot(notebook=True,window_size=[500,500],show_edges=True)

print(f"Voxel similarity: {calculate_voxel_measure(trimesh.load_mesh(simplified_path),trimesh.load_mesh(out_p2))}")


Simplified mesh triangles: 16 -> unique facets: 7
Converged after 9 iterations: alphak=0.000017
Reconstructed convex mesh: 10 vertices, 16 faces


EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

Voxel similarity: 0.9886787307327352
